In [ ]:

import os
import glob
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt


In [ ]:
### METADATA 

obs_cal_mapping_path = "/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/obs_cal_info.csv"

metadata_df = pd.read_csv(obs_cal_mapping_path)


In [ ]:
# find dark signal id & subtract 

def dark_subtract(obs_id, obs_path):
    meta = metadata_df[metadata_df["obs_id"] == obs_id.upper()]
    
    if len(meta) == 0:
        raise ValueError(f"'{obs_id}' not found in obs/cal mapping table.")
    elif len(meta) > 1:
        best_idx = meta["version"].str.extract(r"(\d+)")[0].astype(int).idxmax()
        meta = meta.loc[[best_idx]]
    
    dark_id = meta["dark_signal_id"].iloc[0].lower()
    
    dark_path = os.path.join(dark_dir, f"{dark_id}_l0.fits")
    with fits.open(dark_path) as hdul:
        dark = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]
        
    dark = dark.mean(axis=0) 
    
    with fits.open(obs_path) as hdul:
        obs_image = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]
    
    obs_image = obs_image - dark[np.newaxis, :, :]
    
    return obs_image

### TARGET

In [ ]:
dark_dir = "/home/bekah/m3-pipeline-dev/data/dark/darks_target"
obs_dir = "/home/bekah/m3-pipeline-dev/data/l0_T"

band_defs = {
    "left": slice(9, 15),
}
denom_slice = slice(40, 600)

records = []

obs_files = sorted(glob.glob(os.path.join(obs_dir, "*l0.fits")))
print(f"Found {len(obs_files)} obs files")

for obs_path in obs_files:
    obs_id = os.path.basename(obs_path).replace("_l0.fits", "")
    
    try:
        obs = dark_subtract(obs_id, obs_path)
    except Exception as e:
        print(f"Skipping {obs_id}: {e}")
        continue
    
    denom = np.mean(obs[:, :260, denom_slice], axis=2)  
    
    for band_name, band_slice in band_defs.items():
        num = np.mean(obs[:, :260, band_slice], axis=2)
        ratio = num / denom
        median_at_x = np.nanmedian(ratio, axis=0)  
        
        for x_val, med_val in zip(np.arange(260), median_at_x):
            records.append({
                "obs_id": obs_id,
                "band": band_name,
                "x": x_val,
                "median_ratio": med_val,
            })

df = pd.DataFrame(records)
df.to_csv("median_ratios_per_band_target.csv", index=False)
print(df.head())

In [ ]:
df

In [ ]:
# plot each obs line 
band_defs = {
    "red": slice(9, 15),
}

fig, ax = plt.subplots(figsize=(10, 6))

for band_name in band_defs:
    sub = df[df["band"] == band_name]
    for obs_id, group in sub.groupby("obs_id"):
        ax.plot(group["x"], group["median_ratio"], alpha=0.3, linewidth=1)

ax.set_xlabel("x")
ax.set_ylabel("median ratio")
ax.legend(band_defs.keys())
plt.ylim(-.1,.5)

plt.savefig("target_med_sl_ratio_all.png")

In [ ]:
# median per band across all obs 
df = pd.read_csv("median_ratios_per_band.csv") 

summary = df.groupby(["band", "x"])["median_ratio"].median().reset_index()

for band_name, group in summary.groupby("band"):
    plt.plot(group["x"], group["median_ratio"], linewidth=2, linestyle=":")

plt.xlabel("x")
plt.ylim(-.1,.45)

plt.title("band ratios for target, median of multiple obs") 
plt.xlabel("x")
plt.ylabel("median ratio")

plt.savefig("target_med_sl_ratio.png")

In [ ]:
print(group["median_ratio"].round(4).tolist())


### GLOBAL 

In [ ]:
dark_dir = "/home/bekah/m3-pipeline-dev/data/dark/darks_global"
obs_dir = "/home/bekah/m3-pipeline-dev/data/l0"

band_defs = {
    "left": slice(4, 7),
}
denom_slice = slice(20, 300)

records = []

obs_files = sorted(glob.glob(os.path.join(obs_dir, "*l0.fits")))
print(f"Found {len(obs_files)} obs files")

for obs_path in obs_files:
    obs_id = os.path.basename(obs_path).replace("_l0.fits", "")
    
    try:
        obs = dark_subtract(obs_id, obs_path)
    except Exception as e:
        print(f"Skipping {obs_id}: {e}")
        continue
    
    denom = np.mean(obs[:, :86, denom_slice], axis=2)  
    
    for band_name, band_slice in band_defs.items():
        num = np.mean(obs[:, :86, band_slice], axis=2)
        ratio = num / denom
        median_at_x = np.nanmedian(ratio, axis=0)  
        
        for x_val, med_val in zip(np.arange(86), median_at_x):
            records.append({
                "obs_id": obs_id,
                "band": band_name,
                "x": x_val,
                "median_ratio": med_val,
            })

df = pd.DataFrame(records)
df.to_csv("median_ratios_per_band_global.csv", index=False)
print(df.head())

In [ ]:
df = pd.read_csv("median_ratios_per_band_global.csv") 

In [ ]:
# plot each obs line 
band_defs = {
    "left": slice(4, 7),
}

fig, ax = plt.subplots(figsize=(10, 6))

for band_name in band_defs:
    sub = df[df["band"] == band_name]
    for obs_id, group in sub.groupby("obs_id"):
        ax.plot(group["x"], group["median_ratio"], alpha=0.3, linewidth=1)

ax.set_xlabel("band")
ax.set_ylabel("median ratio")
plt.title("band ratios for multiple global obs") 
ax.legend(band_defs.keys())
plt.ylim(-.1,.5)
plt.tight_layout()

plt.savefig("global_med_sl_ratio_all.png")

In [ ]:
# median per band across all obs 

summary = df.groupby(["band", "x"])["median_ratio"].median().reset_index()

for band_name, group in summary.groupby("band"):
    plt.plot(group["x"], group["median_ratio"], linewidth=2, linestyle=":")
plt.title("band ratios for global, median of multiple obs") 
plt.xlabel("x")
plt.ylabel("median ratio")
plt.axhline(0)
plt.ylim(-.1,.3)

plt.savefig("global_med_sl_ratio.png")

In [ ]:
print(group["median_ratio"].round(4).tolist())
